# 07 · Reporte de margen

**Propósito:** crear las vistas de consumo y materializar el reporte de las naciones con mayor utilidad por año.

**Entradas:** `gold_margin_by_nation_year`, `gold_supplier_profit`, task value `fact_rows` del task `fact_margin`

**Salidas:** vistas `v_margin_report` y `v_top_suppliers` (desde `src/sql/views_gold.sql`), tabla `rpt_margin_top_nations`

In [ ]:
%run ./00_config

In [ ]:
import json
import os

from pyspark.sql import functions as F

from src.utils.spark_helpers import run_sql_file, write_delta

run_sql_file(spark, os.path.join(SQL_DIR, "views_gold.sql"))

## Top naciones por año

In [ ]:
report = (
    spark.table("v_margin_report")
    .filter(F.col("profit_rank") <= cfg.report_top_n)
    .withColumn("run_id", F.lit(RUN_ID))
)
write_delta(report, cfg.table("rpt_margin_top_nations"))

display(report.orderBy("order_year", "profit_rank").limit(20))

## Resumen de la corrida
Fuera de un job (o si se ejecuta desde el orquestador) el task value no existe y se usa `-1`.

In [ ]:
try:
    fact_rows = dbutils.jobs.taskValues.get(
        taskKey="fact_margin", key="fact_rows", default=-1, debugValue=-1
    )
except Exception as exc:  # el task fact_margin no existe en este contexto
    log.warning("fact_rows task value not available: %s", exc)
    fact_rows = -1

summary = {
    "report_rows": spark.table(cfg.table("rpt_margin_top_nations")).count(),
    "fact_rows": fact_rows,
}
log.info("Report summary: %s", summary)
dbutils.notebook.exit(json.dumps(summary))